# 36. Run a stream with the plasticity indicators attached

This notebook learns one eight-task stream and measures the plasticity
indicators while it learns. It is the only way to get those numbers: four of
the five are read at a task boundary, and the paired study keeps two rolling
resume slots rather than one checkpoint per task, so a finished run has no
boundaries left to read.

### What rides along

| Measurement | Where it comes from |
| --- | --- |
| `delta_w_target`, `norm_target` | the target weights the generator writes, through one fixed reference code |
| `delta_w_generator`, `norm_generator` | the generator's own parameters |
| `effective_rank`, `srank` | the classifier's input features, on three fixed image sets |
| `dormant_0.01`, `dormant_0.025`, `dormant_0.1` | ReLU units that never respond, on the same three sets |
| `loss_progress` | how far one update moves the loss, sampled every tenth update |

Each one is written in `uncle/plasticity_indicators.py` and tested there. This
notebook only supplies the images and attaches the observer.

### What it costs

One source job: eight tasks, five epochs each, on the paired study's own
settings. The indicators add one extra forward pass in every ten updates plus a
few forward passes at each task boundary, which is a few percent of the total.
Give the session a real GPU and a budget you are willing to spend. An
interrupted session resumes from the last verified R2 save and the indicator
file picks up at the same place.

### What it does not touch

Nothing here changes `paired_generalization_v2`. This notebook creates its own
study under its own name, so the frozen study and its recorded code revision
stay exactly as they are. Notebooks 19 and 20 are untouched too.

## 1. Load the code

In [ ]:
from pathlib import Path
import sys, os, json, time, shutil, subprocess, importlib.util

IN_COLAB = "google.colab" in sys.modules
if any(name == "uncle" or name.startswith("uncle.") for name in sys.modules):
    raise RuntimeError("Restart the runtime before loading the code.")

# This notebook runs at the current main, not at a pinned revision. The
# indicators were written after the paired study was frozen, so a pinned
# checkout would not contain them.
if IN_COLAB:
    REPO = Path("/content/path-indicators")
    if not REPO.exists():
        subprocess.run(["git", "clone", "https://github.com/sumitasthana/PATH.git", str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    REPO = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "uncle/task_partition.json").exists()), None)
    if REPO is None:
        raise FileNotFoundError("Open this notebook from a PATH checkout.")

for package in ("boto3", "tqdm", "matplotlib"):
    if importlib.util.find_spec(package) is None:
        if not IN_COLAB:
            raise ImportError("Install the storage and extras dependencies first.")
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", package], check=True)
sys.path.insert(0, str(REPO))

from uncle.storage import R2Store
from uncle.paired_study import StudyStore, make_plan, jobs, run_job
from uncle.plasticity_indicators import StreamObserver, fixed_image_sets, indicator_key
from uncle.streams import build_tasks

import torch

HEAD = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("Running at:", HEAD)
print("PyTorch:", torch.__version__, "| CUDA build:", torch.version.cuda)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU. This notebook trains; switch to a GPU runtime before section 7.")

## 2. Settings

In [ ]:
# The study name. This is a new study, separate from paired_generalization_v2,
# so nothing here can disturb the frozen one. Keep the name if you add seeds
# later: the settings are checked against the saved copy and a difference stops
# the notebook rather than silently mixing two protocols.
STUDY_ID = "indicator_stream_v1"
BUCKET = "uncle-experiments"

ORDER = "order_01"            # tasks 3, 0, 9, 5, 17, 1, 7, 14
SEED = 0                      # one seed per session; 0, 1 and 2 are planned

# The held-out Tiny ImageNet task. It is never learned in this order, so its
# images show what the generator does to data it has not been trained on. Task
# 2 is the same probe task notebooks 29 and 35 use, which keeps the numbers
# comparable with the probe results already recorded.
PROBE_TASK = "2"

# Images per fixed set. The sets are fixed because the adapter plasticity study
# first measured rank and dormant units on the current task's images, then had
# to redo the whole analysis: measuring on moving data confuses "the
# representation of this data changed" with "the representation changed".
IMAGES_PER_SET = 256
INCLUDE_EUROSAT = True        # the third set, from a different dataset entirely

# A soft session budget. The run stops at the next verified R2 save once the
# budget is gone, so nothing is lost and the next session resumes there.
MAX_SESSION_MINUTES = 240
SAVE_RESERVE_MINUTES = 15

RELEASE_GPU_WHEN_DONE = False

DATA_ROOT = Path("/content/data" if IN_COLAB else REPO / "data")
EUROSAT_ROOT = Path("/content/eurosat" if IN_COLAB else REPO / "data/eurosat")
if not 0 < SAVE_RESERVE_MINUTES < MAX_SESSION_MINUTES:
    raise ValueError("The reserve must be smaller than the session budget.")
print("Study:", STUDY_ID, "| order:", ORDER, "| seed:", SEED)

## 3. Connect to R2

In [ ]:
def secret(name):
    value = os.environ.get(name)
    if not value and IN_COLAB:
        from google.colab import userdata
        value = userdata.get(name)
    if not value:
        raise RuntimeError("Missing secret: " + name)
    return value

store = R2Store.connect(secret("R2_ENDPOINT"), BUCKET,
    secret("R2_ACCESS_KEY_ID"), secret("R2_SECRET_ACCESS_KEY"))
store.progress = True
CACHE = REPO / "outputs/paired_study" / STUDY_ID
CACHE.mkdir(parents=True, exist_ok=True)

# Where this run's boundary record lives. The JSONL holds one line per task
# boundary. The .baseline.pt beside it holds the two weight vectors from
# boundary zero, which is what relative weight change is measured against: once
# the generator has moved, boundary zero cannot be recreated from anything else
# the run saves, so a resume without it would silently report no movement.
LOG_KEY = indicator_key(STUDY_ID, ORDER, SEED)
BASELINE_KEY = LOG_KEY.replace(".jsonl", ".baseline.pt")
LOG_PATH = CACHE / "indicators" / f"{ORDER}_seed_{SEED}.jsonl"
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)

print("Connected to R2 bucket:", BUCKET)
print("Indicator record:", LOG_KEY)

## 4. Open or create the study

In [ ]:
# A study saved earlier is reopened as it stands. Only a first run writes a new
# plan, and `prepare` refuses to overwrite settings that differ, so two seeds of
# the same study can never be trained under two different configurations.
STUDY_KEY = "uncle/paired_generalization/" + STUDY_ID + "/study.json"
if store.info(STUDY_KEY) is not None:
    plan = json.loads(store.download(STUDY_KEY, CACHE / "cache").read_text(encoding="utf-8"))
    print("Reopened the saved study, recorded at code revision", plan["code_version"][:12])
    if plan["code_version"] != HEAD:
        print("This checkout is at", HEAD[:12],
              "\nThe training settings still match; only the recorded revision differs.")
else:
    plan = make_plan(HEAD, REPO / "uncle/task_partition.json", study_id=STUDY_ID)
    print("Created a new study at code revision", HEAD[:12])

storage = StudyStore(store, plan, CACHE, cleanup_completed=False)
storage.prepare()

# The settings are the paired study's own, so the indicators describe a stream
# trained exactly the way the recorded results were trained.
settings = plan["config"]
print("Backbone:", settings["backbone"], "| epochs per task:", settings["epochs"],
      "| batch:", settings["batch_size"], "| learning rate:", settings["learning_rate"])

ORDER_TASKS = list(plan["orders"][ORDER])
JOB = next(j for j in jobs(plan) if j["id"] == f"sources/{ORDER}/seed_{SEED}")
print("Job:", JOB["id"])
print("Tasks, in order:", " ".join(ORDER_TASKS))
if PROBE_TASK in ORDER_TASKS:
    raise ValueError("The probe task is in the learning order, so it is not held out.")
if storage.completed_report(JOB) is not None:
    print("\nThis job is already finished in R2. Section 7 will return its report "
          "without training. If its indicator file is missing, the boundaries are "
          "gone and the job would have to be rerun under a new study name.")

## 5. Build the images

In [ ]:
# One call builds every task this session needs. Indexing the dataset is the
# largest cost before the first optimizer step, so the probe task is built here
# with the training tasks rather than in a second pass. The probe task is never
# trained; it only supplies images.
needed = tuple(ORDER_TASKS + [PROBE_TASK])
tasks = build_tasks(root=DATA_ROOT, partition_path=REPO / "uncle/task_partition.json",
                    partition_seed=42, include=needed, download=True)
print("Loaded Tiny ImageNet images for", len(needed), "tasks")

from torch.utils.data import ConcatDataset

sources = {
    # Held-out images of the tasks this stream learns. These say what happens
    # to the representation of data the model is being trained on.
    "in_stream": ConcatDataset([tasks[name]["test"] for name in ORDER_TASKS]),
    # A Tiny ImageNet task the stream never sees. Same dataset, unseen classes.
    "probe": tasks[PROBE_TASK]["test"],
}

if INCLUDE_EUROSAT:
    from torchvision.datasets import EuroSAT
    from torchvision.transforms import ToTensor
    # Satellite images, already 64x64, which is what the backbone expects. A
    # measure that moves on Tiny ImageNet and stays flat here is saying
    # something different from one that moves on both.
    eurosat = EuroSAT(root=str(EUROSAT_ROOT), transform=ToTensor(), download=True)
    assert eurosat[0][0].shape == (3, 64, 64), f"Unexpected shape {eurosat[0][0].shape}"
    sources["eurosat"] = eurosat

# The same pictures at every boundary, in every run, for every seed and order,
# because the draw is seeded inside the library and not here. Only the model
# differs between two measurements.
image_sets = fixed_image_sets(sources, count=IMAGES_PER_SET)
for name, images in image_sets.items():
    print(f"{name:<10} {tuple(images.shape)}")

## 6. Pick up an interrupted run

In [ ]:
# Both files come back together or not at all. The JSONL alone would raise at
# the first boundary, because weight change has nothing to measure against.
if store.info(LOG_KEY) is not None:
    shutil.copyfile(store.download(LOG_KEY, CACHE / "cache"), LOG_PATH)
    baseline = LOG_PATH.with_suffix(".baseline.pt")
    if store.info(BASELINE_KEY) is None:
        raise RuntimeError(
            f"{LOG_KEY} is in R2 but {BASELINE_KEY} is not. Weight change is "
            "measured against boundary zero, which cannot be recreated. Start a "
            "new study name rather than continuing against a new baseline.")
    shutil.copyfile(store.download(BASELINE_KEY, CACHE / "cache"), baseline)
    written = [json.loads(line) for line in
               LOG_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
    print(f"Resuming: {len(written)} boundaries already measured, "
          f"up to t = {max(row['t'] for row in written)}.")
else:
    LOG_PATH.unlink(missing_ok=True)
    LOG_PATH.with_suffix(".baseline.pt").unlink(missing_ok=True)
    print("Starting fresh: no indicator record in R2 yet.")

## 7. Run the stream with the indicators attached

The run saves its state more often than once per task: at every epoch and at
the start of each request. The observer does not need to know which of those is
a task boundary, because boundaries are numbered by how many tasks have been
learned. A number already in the file is returned from before anything is
measured, so epoch saves cost nothing and the first call at each number is
always the save that follows a completed task.

Boundary zero is measured before the first update, on the fresh generator.

In [ ]:
class UploadingObserver(StreamObserver):
    """The observer, plus an upload of both files whenever a boundary lands.

    The Colab disk disappears when the session ends, so a record that only
    lives there is lost with it. Uploading at each boundary means an
    interrupted session keeps every task it finished.
    """

    def boundary(self, trainer, t, task):
        row = super().boundary(trainer, t, task)
        if row is not None:
            store.upload(self.log.baseline_path, BASELINE_KEY, overwrite=True)
            store.upload(self.log.path, LOG_KEY, overwrite=True)
            print(f"  boundary t={t} after task {task}: "
                  f"delta_w_generator {row['delta_w_generator']:.4f}, "
                  f"effective_rank_in_stream {row['effective_rank_in_stream']:.2f}")
        return row


observer = UploadingObserver(LOG_PATH, image_sets,
                             study=STUDY_ID, order=ORDER, seed=SEED)

deadline = time.monotonic() + (MAX_SESSION_MINUTES - SAVE_RESERVE_MINUTES) * 60
store.progress = False     # one bar per transfer would bury the training bars
began = time.monotonic()
try:
    report = run_job(plan, JOB, tasks, storage, device="cuda", progress=True,
                     stop_requested=lambda: time.monotonic() >= deadline,
                     observer=observer)
    print("\nStatus:", report["status"])
    print(f"Elapsed: {(time.monotonic() - began) / 60:.1f} minutes")
    if report["status"] == "complete":
        print("Final accuracies:", report["final_accuracies"])
    else:
        print("Budget reached at a verified save. Run this notebook again with the "
              "same settings to continue.")
finally:
    store.progress = True
    rows = [json.loads(line) for line in
            LOG_PATH.read_text(encoding="utf-8").splitlines() if line.strip()] \
        if LOG_PATH.exists() else []
    print(f"Indicator record: {len(rows)} boundaries in {LOG_KEY}")
    del tasks
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if RELEASE_GPU_WHEN_DONE and IN_COLAB:
        from google.colab import runtime
        print("Releasing the Colab runtime.")
        runtime.unassign()

## 8. What comes next

A finished run leaves nine lines in `uncle/indicators/`: boundary zero plus one
per task. Repeat this notebook with `SEED = 1` and `SEED = 2` to get a spread,
then open notebook 37 on a CPU runtime to read the files, build the table and
draw the curves. Notebook 37 downloads records, never models, so it needs no
GPU.